# Week 3: MLP protein context

This original notebook reads only the committed aggregate Week 3 public report. It does not train, load a collection, open a checkpoint, or access sealed test data. All scores are native-validation evidence. Embedding plots and cosine summaries are descriptive diagnostics, not biological mechanism or function evidence.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

candidates = [
    Path.cwd() / "reports/week_03/mlp_evaluation_v1.json",
    Path.cwd() / "../../reports/week_03/mlp_evaluation_v1.json",
]
report_path = next((candidate.resolve() for candidate in candidates if candidate.exists()), None)
if report_path is None:
    raise FileNotFoundError("Run from the repository root or notebooks/week_03 after publishing the Week 3 report.")
report = json.loads(report_path.read_text(encoding="utf-8"))
assert report["scope"] == "week_03_mlp_public_report"
report["hard_gates"]

## The teaching model

For each target, the model performs **lookup → flatten → tanh hidden → logits**. It looks up the last C residue tokens, flattens their embeddings, applies a learned affine map and `tanh`, then projects to 21 logits for the 20 residues plus EOS. For C=20 and E=32, the flattened width is 640. The parameter count is `21×32 + 640×800 + 800 + 800×21 + 21 = 530,293`.

A causal PLM is a statistical factorization. A ribosome reads mRNA codons, so this is not a claim that biology selects a next residue from previously observed amino acids.

In [ ]:
model = report["model"]["context20"]
terms = {
    "embedding": 21 * model["embedding_width"],
    "input_to_hidden": model["context_length"] * model["embedding_width"] * model["hidden_width"],
    "hidden_bias": model["hidden_width"],
    "hidden_to_logits": model["hidden_width"] * 21,
    "logit_bias": 21,
}
terms, sum(terms.values()), model["parameter_count"]

In [ ]:
curve = report["learning_curves"]["series"]
for series in curve:
    plt.plot([row["prediction_position"] / 1e6 for row in series["points"]], [row["mean_cross_entropy"] for row in series["points"]], marker="o", label=series["model"])
plt.xlabel("training predictions (millions)")
plt.ylabel("native-validation cross-entropy")
plt.legend()
plt.title("Frozen C10 and C20 learning curves")
plt.show()

In [ ]:
final = report["final_three_seed_comparison"]
baseline = report["fixed_budget_baseline_comparison"]
labels = ["C20", "E64", "Week 2 bigram"]
values = [final["context20"]["aggregate"]["mean_cross_entropy"], final["embedding64_challenger"]["aggregate"]["mean_cross_entropy"], baseline["baseline"]["cross_entropy"]]
plt.bar(labels, values)
plt.ylabel("native-validation CE")
plt.title("Fixed-budget validation comparison")
plt.show()

In [ ]:
panels = report["embedding_diagnostics"]["pca_coordinates"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for axis, panel in zip(axes, panels, strict=True):
    for point in panel["coordinates"]:
        axis.scatter(point["pc1"], point["pc2"], s=20)
        axis.annotate(point["token"], (point["pc1"], point["pc2"]), fontsize=8)
    axis.set_title(f"seed {panel['seed']}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
fig.suptitle("Separate centered-SVD PCA panels: axes are not aligned across seeds")
plt.show()

In [ ]:
pairs = sorted(report["embedding_diagnostics"]["residue_cosine_similarity"]["pairs"], key=lambda row: row["mean_cosine_similarity"], reverse=True)[:10]
plt.errorbar([row["residue_pair"] for row in pairs], [row["mean_cosine_similarity"] for row in pairs], yerr=[row["sample_standard_deviation"] for row in pairs], fmt="o")
plt.xticks(rotation=45)
plt.title("Top within-seed residue cosine pairs")
plt.show()

In [ ]:
bins = report["position_availability_diagnostic"]["bins"]
labels = [row["bin"] for row in bins]
plt.bar(labels, [row["embedding64_minus_context20_cross_entropy"] for row in bins])
plt.xticks(rotation=25)
plt.ylabel("E64 minus C20 CE")
plt.title("Descriptive position-bin CE difference")
plt.show()

## Limits

The 25M screen, 100M final comparison, and post-freeze position diagnostic are separate evidence stages. H=1600 was only screened at 25M, not run at 100M. Remaining cross-entropy can reflect genuine conditional variability plus missing family, function, global-fold, distant-residue, or future-context information. This notebook makes no training, tuning, test, significance, mechanism, structure, or function claim.